# OminiControl2 · 阶段一冒烟测试(KV-Cache)一键 Notebook

**用途**:在远程 GPU 机器上从上到下依次运行所有 cell,完成阶段一冒烟测试——
验证 FLUX 加载 → 多卡 dispatch → LoRA 挂载 → `generate()` → baseline vs kv_cache 计时全链路。

**针对本机(8×RTX 4090, 24 GB)已内置的修复**(细节见 `repro/TROUBLESHOOTING.md`):
- FLUX.1-dev bf16 ≈33 GB 单卡放不下 → **2 卡手工 dispatch**:cuda:0 = encoders+vae+latents(主场),cuda:1 = transformer 整块;
- 在 `omini.transformer_forward` 入口装**跨卡桥**:输入搬进 cuda:1,**输出搬回 cuda:0**(修复 scheduler.step / vae.decode 跨卡);
- LoRA 加载后做 **device sweep**(修复 PEFT 把 LoRA 权重留在 cuda:0 的失败 #11)。

**运行前提**:`conda activate omini`(或 `source train/setup_env.sh`)后用该环境的 Jupyter 内核打开本 notebook。

**预计耗时**:模型已缓存的情况下约 5–10 分钟。

## 0. 环境自检 — 任何一项不对都在这里报错,而不是深入管线后才炸

In [ ]:
import os
import sys
import importlib

def fail(msg, section):
    raise RuntimeError(f"[环境自检失败] {msg}\n→ 处理办法见 repro/TROUBLESHOOTING.md {section}")

# ---- 0.1 定位仓库根(notebook 可放在仓库根或 repro/ 下运行) ----
REPO_ROOT = None
for cand in (os.getcwd(), os.path.dirname(os.getcwd())):
    if os.path.exists(os.path.join(cand, "omini", "pipeline", "flux_omini.py")):
        REPO_ROOT = os.path.abspath(cand)
        break
if REPO_ROOT is None:
    fail("找不到 OminiControl 仓库根,请在仓库根或 repro/ 目录下启动 Jupyter", "§1")
sys.path.insert(0, REPO_ROOT)
sys.path.insert(0, os.path.join(REPO_ROOT, "repro"))
os.chdir(REPO_ROOT)
print(f"REPO_ROOT = {REPO_ROOT}")

# ---- 0.2 Python / 关键包版本(远程机器失败 #1 的教训:版本必须先卡死) ----
if sys.version_info < (3, 10):
    fail(f"Python {sys.version.split()[0]} 太旧(系统 3.8 的坑),需要 omini conda 环境的 3.12。"
         f"当前内核 = {sys.executable}", "§1.1")
print(f"Python {sys.version.split()[0]}  内核: {sys.executable}")

expected = {
    "torch":        lambda v: v.startswith("2."),
    "diffusers":    lambda v: v == "0.38.0",   # 其它版本会破 KV-Cache 的 cache_idx
    "transformers": lambda v: v.startswith("4.") and int(v.split(".")[1]) >= 55,
    "huggingface_hub": lambda v: int(v.split(".")[0]) == 0,   # 必须 <1.0
    "peft":         lambda v: True,
    "cv2":          lambda v: True,
}
for pkg, ok in expected.items():
    try:
        mod = importlib.import_module(pkg)
    except ImportError as e:
        fail(f"缺包 {pkg}: {e}(确认内核是 omini 环境)", "§1")
    ver = getattr(mod, "__version__", "?")
    if not ok(ver):
        fail(f"{pkg}=={ver} 版本不符合要求", "§1")
    print(f"{pkg:>18} == {ver}")

# ---- 0.3 GPU ----
import torch
if not torch.cuda.is_available():
    fail("torch.cuda 不可用(驱动/wheel 不匹配?)", "§1.2")
n_gpu = torch.cuda.device_count()
for d in range(n_gpu):
    p = torch.cuda.get_device_properties(d)
    free, total = torch.cuda.mem_get_info(d)
    print(f"GPU {d}: {p.name} | 总 {total/1024**3:.1f} GB | 空闲 {free/1024**3:.1f} GB")
free0, _ = torch.cuda.mem_get_info(0)
free1 = torch.cuda.mem_get_info(1)[0] if n_gpu > 1 else 0
if free0 / 1024**3 < 13 or (n_gpu > 1 and free1 / 1024**3 < 23):
    fail("cuda:0 需要约 12 GB 空闲(encoders+vae),cuda:1 需要约 23 GB 空闲(transformer 整块)。"
         "先 nvidia-smi 清掉占卡进程。", "§2")

# ---- 0.4 HF 鉴权 / 模型缓存(FLUX.1-dev 是 gated) ----
from huggingface_hub import get_token
cache_hit = any(
    os.path.isdir(os.path.expanduser(f"~/.cache/huggingface/hub/models--{m}"))
    for m in ("black-forest-labs--FLUX.1-dev",)
)
if not cache_hit and get_token() is None:
    fail("FLUX.1-dev 无本地缓存且未登录 HF(gated 模型会 401)。"
         "先在网页同意协议,再 huggingface-cli login", "§6")
print(f"FLUX 本地缓存: {'命中' if cache_hit else '未命中(将联网下载 ~36 GB)'} | HF token: {'已配置' if get_token() else '无'}")

print("\n✅ 环境自检全部通过")

## 1. 参数配置(冒烟测试默认值,一般不用改)

In [ ]:
FLUX_PATH      = "black-forest-labs/FLUX.1-dev"
LORA_REPO      = "Yuanshi/OminiControl"              # 阶段三换成 runs/<run名>/ckpt
LORA_WEIGHT    = "experimental/canny.safetensors"    # 阶段三换成训练产物文件名
ADAPTER        = "canny"
CONDITION_TYPE = "canny"
IMAGE          = "assets/vase_hq.jpg"
PROMPT         = "A beautiful vase on a wooden table."
STEPS          = 8
N_COND         = 1
REPEATS        = 2
SEED           = 42
GUIDANCE       = 3.5        # FLUX.1-dev 蒸馏值,勿动
SIZE           = 512
DISPATCH       = "auto"     # auto: 单卡>=40GB 用 single,否则 2gpu(本机 4090 会走 2gpu)
OUT            = "repro/kvcache_results"

print("配置就绪。冒烟测试口径:单条件 8 步,速度参考值 ≈1.5x,质量不作数(v1 权重非独立训练)。")

## 2. 加载 pipeline(含 2 卡 dispatch + 跨卡桥)+ LoRA(含 device sweep)

首次运行约 1–3 分钟(权重已缓存时)。日志里应看到:
`dispatch 完成` → `已安装 tx bridge` → `device 安全网:搬了 N 个 param/buffer`。

In [ ]:
from kvcache_benchmark import load_pipeline, attach_lora, build_conditions, time_one, sweep_devices

# WHY 用 globals() 守卫:重复运行本 cell 时不重新加载 36 GB 模型
if "pipe" not in globals():
    pipe = load_pipeline(FLUX_PATH, dispatch=DISPATCH)
else:
    print("[跳过] pipe 已加载(重复运行本 cell 属正常)")

try:
    attach_lora(pipe, LORA_REPO, LORA_WEIGHT, ADAPTER)
except ValueError as e:
    # PEFT 对重复的 adapter_name 会抛 ValueError;重跑 cell 时只需重新激活 + sweep
    if "already" in str(e).lower() or "exist" in str(e).lower():
        print(f"[跳过] adapter '{ADAPTER}' 已加载,重新激活")
        pipe.set_adapters([ADAPTER])
        sweep_devices(pipe)
    else:
        raise

print("\n✅ pipeline 就绪")

## 3. 冒烟测试:baseline vs kv_cache 各跑一遍并计时

每个开关先 warmup 1 次(不计时,吃掉 kernel 编译开销),再计时 `REPEATS` 次取中位。
8 步 × (1 warmup + 2 计时) × 2 个开关,4090 上预计 3–6 分钟。

In [ ]:
import torch
from omini.pipeline.flux_omini import generate

conditions = build_conditions(N_COND, IMAGE, CONDITION_TYPE, ADAPTER, SIZE)

def make_gen(kv):
    def gen():
        # generator 固定在主场 cuda:0:latents 由它产出,必须与 scheduler 同卡
        g = torch.Generator(device="cuda:0").manual_seed(SEED)
        return generate(
            pipe, prompt=PROMPT, conditions=conditions,
            height=SIZE, width=SIZE, num_inference_steps=STEPS,
            guidance_scale=GUIDANCE, generator=g, kv_cache=kv,
        ).images[0]
    return gen

results = {}
for kv in (False, True):
    print(f"—— 测量 kv_cache={kv} ——")
    try:
        med, std, peak, img = time_one(make_gen(kv), REPEATS)
    except Exception as e:
        raise RuntimeError(
            f"kv_cache={kv} 生成失败:{type(e).__name__}: {e}\n"
            "→ 若是 device 类报错,按 repro/TROUBLESHOOTING.md §3/§4 分诊") from e
    results[kv] = {"med": med, "std": std, "peak": peak, "img": img}
    print(f"   {med:.2f}s ± {std:.2f} | 峰值显存 {peak:.1f} GB")

speedup = results[False]["med"] / results[True]["med"]
print(f"\n{'='*50}\n加速比: {speedup:.2f}x  (单条件 {STEPS} 步;README 参考值 ≈1.5x @ 8 步)\n{'='*50}")
if speedup < 1.1:
    print("⚠️ 加速比 < 1.1x:确认 kv_cache 真的生效(TROUBLESHOOTING.md §6 cache_idx 一节)")

## 4. 保存并预览两张生成图(左 = baseline,右 = kv_cache)

In [ ]:
from PIL import Image

os.makedirs(OUT, exist_ok=True)
results[False]["img"].save(os.path.join(OUT, f"smoke_c{N_COND}_s{STEPS}_base.png"))
results[True]["img"].save(os.path.join(OUT, f"smoke_c{N_COND}_s{STEPS}_kv.png"))
print(f"已保存到 {OUT}/")

combo = Image.new("RGB", (SIZE * 2 + 8, SIZE), "white")
combo.paste(results[False]["img"], (0, 0))
combo.paste(results[True]["img"], (SIZE + 8, 0))
combo

## 5. 结果判读 & 下一步

| 观察项 | 冒烟测试口径 |
|---|---|
| 两个开关都跑完、无异常 | ✅ 阶段一通过 |
| 加速比 ≈ 1.2–1.6x | ✅ 正常(单条件 8 步) |
| 左右两图差异明显 | **属预期**——v1 权重不是 `independent_condition` 训练的,质量对比要等阶段二训出的 LoRA |

**下一步(阶段二/三,见 `repro/REPRODUCE_FEATURE_REUSE.md`)**:
1. `bash train/script/train_feature_reuse.sh` 训 `independent_condition: true` 的 LoRA;
2. 把上面 cell 1 的 `LORA_REPO/LORA_WEIGHT` 换成训练产物,`STEPS/N_COND` 扫 `8,20,28 × 1,2,3`,或直接用 CLI:
   ```bash
   python repro/kvcache_benchmark.py --lora-repo runs/<run名>/ckpt \
     --lora-weight <权重>.safetensors --adapter-name canny --condition-type canny \
     --steps 8,20,28 --conditions 1,2,3 --repeats 3
   ```

**报错了?** 按 `repro/TROUBLESHOOTING.md` 的"快速分诊表"用报错关键词定位章节。